In [ ]:
%pip install langchain langchain-community langchain-groq langchain-huggingface langchain-chroma sentence-transformers

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
import json
from google.colab import userdata

In [3]:
# Retrieve the secret by the name you gave it
groq_api_key = userdata.get('GROQ_API_KEY')

# Now you can use it in your logic
print("API Key loaded successfully!")

API Key loaded successfully!


In [4]:
loader = TextLoader("compound_data_descriptions.txt")
docs = loader.load()  # Returns list of Document objects

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # Max characters per chunk
    chunk_overlap=200,   # Overlap for context
    add_start_index=True # Track original position
)
chunks = splitter.split_documents(docs)

In [ ]:
# Embeddings - local Qwen model (downloads on first use ~1.19G)
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"trust_remote_code": True}
)

In [6]:
# Persist to Chroma - best for local dev/production similarity search
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./compound_information_db"  # Saves locally
)

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})  # Top 5 chunks

# Fast Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=groq_api_key,
    temperature=0,
    max_tokens=None)

# Real Estate Specialist RAG prompt
compound_prompt = ChatPromptTemplate.from_template("""
You are an expert Real Estate Consultant specializing in the Egyptian market.
Your goal is to provide accurate, professional, and helpful information about compounds and properties based on the data provided.

Strict Guidelines:
1. Answer ONLY using the context below. Do not invent amenities or prices.
2. If the context doesn't mention a specific detail (like 'delivery date' or 'down payment'), clearly state that it is not specified.
3. Highlight key selling points like location, developer reputation, and unique facilities.
4. Keep the tone professional, persuasive, and data-driven.

Context:
{context}

Question: {question}

Answer:"""
)

# Simple 2-step RAG chain: retrieve → format → prompt → llm → parse
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

compound_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | compound_prompt
    | llm
    | StrOutputParser()
)

In [18]:
# Usage
query = "Who are the three developers behind the Jirian - Nations of Sky project?"
response = compound_chain.invoke(query)
print(response)

The three developers behind the Jirian - Nations of Sky project are Nations of Sky, Palm Hills, and Mountain View.


In [17]:
# Usage
query = "How long does it take to drive from Jirian - Nations of Sky to Sphinx International Airport?"
response = compound_chain.invoke(query)
print(response)

According to the provided information, it takes approximately 18 minutes to drive from Jirian - Nations of Sky to Sphinx International Airport.


In [16]:
# Usage
query = "What types of residential units are available for sale in Jirian - Nations of Sky?"
response = compound_chain.invoke(query)
print(response)

Based on the provided context, the types of residential units available for sale in Jirian by Nations of Sky are:

- Standalone villas
- Townhouses
- Twin houses
- Penthouses
- Apartments
- Duplexes

These various types of residential units cater to different tastes and preferences, offering a range of options for potential buyers.


**"Negative" Tests (To ensure the LLM doesn't hallucinate)**

In [14]:
# Usage
query = "What is the down payment percentage and installment plan for a townhouse in Jirian - Nations of Sky?"
response = compound_chain.invoke(query)
print(response)

Unfortunately, the provided context does not specify the down payment percentage and installment plan for a townhouse in Jirian by Nations of Sky. This information is not mentioned in the FAQs or the description of the compound.

However, I can highlight some key selling points of Jirian by Nations of Sky:

* **Prime Location**: Jirian by Nations of Sky is located in New Zayed, on the 26th of July Axis, offering easy access to major roads and attractions.
* **Developer Reputation**: The compound is developed by Nations of Sky, in partnership with Mountain View and Palm Hills, two well-known and reputable developers in the Egyptian market.
* **Unique Facilities**: Jirian by Nations of Sky features a wide range of recreational facilities, social hubs, and unique experiences, such as the Nile 180 and Blue Islands, which offer residents a fun and luxurious lifestyle.
* **Variety of Properties**: The compound offers a wide selection of lavish units, including standalone villas, townhouses, 

In [15]:
# Usage
query = "What is the starting price for a 3-bedroom apartment in Jirian - Nations of Sky?"
response = compound_chain.invoke(query)
print(response)

Based on the provided information, the starting price for apartments in Jirian by Nations of Sky is not specified for a 3-bedroom apartment specifically. However, the listed prices for apartments start from 8,097,153 EGP.

It's worth noting that the prices mentioned are for apartments in general, and the actual price for a 3-bedroom apartment may vary depending on factors such as size, location, and amenities. I would recommend contacting the developer or a real estate agent for more information on the specific prices and availability of 3-bedroom apartments in Jirian by Nations of Sky.

Key selling points of Jirian by Nations of Sky include:

* Prime location in New Zayed, on 26th of July Axis
* Unobstructed views of the River Nile
* Partnership with reputable developers Mountain View, Palm Hills, and Nations of Sky
* Wide selection of lavish units, including standalone villas, townhouses, twin houses, penthouses, apartments, and duplexes
* Close proximity to upscale residential commu

In [19]:
!zip -r compound_information_db.zip compound_information_db

  adding: compound_information_db/ (stored 0%)
  adding: compound_information_db/65e96e3b-b835-4c37-8298-6dbfefe8bbb5/ (stored 0%)
  adding: compound_information_db/65e96e3b-b835-4c37-8298-6dbfefe8bbb5/header.bin (deflated 55%)
  adding: compound_information_db/65e96e3b-b835-4c37-8298-6dbfefe8bbb5/index_metadata.pickle (deflated 45%)
  adding: compound_information_db/65e96e3b-b835-4c37-8298-6dbfefe8bbb5/length.bin (deflated 99%)
  adding: compound_information_db/65e96e3b-b835-4c37-8298-6dbfefe8bbb5/data_level0.bin (deflated 53%)
  adding: compound_information_db/65e96e3b-b835-4c37-8298-6dbfefe8bbb5/link_lists.bin (deflated 78%)
  adding: compound_information_db/chroma.sqlite3 (deflated 52%)


In [20]:
from google.colab import files
files.download("compound_information_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>